In [34]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [35]:
df_train = pd.read_csv('../../data/train.csv')
print(f'Dimensions of DataFrame: {df_train.shape}')
df_train.head(5)

Dimensions of DataFrame: (3000888, 6)


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [36]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB


In [37]:
# Change type of columns
df_train['date'] = pd.to_datetime(df_train['date'])
df_train['family'] = df_train['family'].astype(str)

In [38]:
df_test = pd.read_csv('../../data/test.csv')
print(f'Dimensions of DataFrame: {df_test.shape}')
df_test.head(5)

Dimensions of DataFrame: (28512, 5)


,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2
3,3000891,2017-08-16,1,BEVERAGES,20
4,3000892,2017-08-16,1,BOOKS,0


In [39]:
df_test['date'] = pd.to_datetime(df_test['date'])
df_test['family'] = df_test['family'].astype(str)

We have to predict for each of the ids the number of sales

In [40]:
n_families = len(df_train['family'].unique())
print(f"There are {n_families} different families")

There are 33 different families


In [41]:
df_train_1 = df_train.loc[df_train['store_nbr'] == 1]
fig = px.line(df_train_1, x = 'date', y = 'sales', color = 'family')
fig.show()

We can appreciate that the time series aren't in the same scale, and that some families don't have any sale (BABY CARE).

We also can appreciate that the majority of families have weekly cycles.

Some of the families have periods without any sale and outliers:
- Books (any sale since mid 2016)
- Celebration (periods without any sale and an outlier in mid 2014)
- Frozen Foods (outlier in finals of 2015)
- Home and Kitchen I (outlier in beginning of 2016)
- Home and Kitchen II (periods without sales)
- Home Appliances (periods without sales and outlier mid 2016)
- Home Care (periods without sales)
- Ladieswear (periods without sales)
- Magazines (periods without sales)
- Personal Care (outliers mid 2016 and beginning 2017)
- Pet supplies (Periods without sales)
- Players and electronics (Periods without sales)
- Produce (Periods without sales)
- School and Office Supplies (Periods without sales)

In [42]:
# To observe better the histogram we scale it
df_scale = df_train_1.pivot(columns = 'family', values = 'sales', index = ['date', 'store_nbr'])
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_scale)
df_scale = pd.DataFrame(scaled_data, index = df_scale.index, columns = df_scale.columns).reset_index()
df_scale = pd.melt(df_scale, id_vars=['date', 'store_nbr'], var_name='family', value_name='sales')
fig = px.histogram(df_scale, x = 'sales', color = 'family')
fig.show()

We can see that some families have weekly stationality (PREPARED FOODS and MEATS) and some not

In [44]:
# Data processing of outliers and missing periods
# Outliers -> Average
# Missing periods -> Inerpolate except if the missing period is in the beginning, then drop

In [45]:
n_stores = len(df_train['store_nbr'].unique())
print(f"There are {n_stores} different stores")

There are 54 different stores


In [46]:
# Do all the stores have the same families?
df_families = df_train.groupby(by = ['family'], as_index=False)['store_nbr'].nunique()
df_families.columns = ['family', 'number_stores']
fig = px.bar(df_families, x = 'family', y = 'number_stores')
fig.show()

We can see that all of the families are in the 54 stores

In [47]:
print(f"If we do one model for each tuple store-family, we have a total of {n_stores * n_families} models")

If we do one model for each tuple store-family, we have a total of 1782 models


In [48]:
init_test = df_test['date'].min()
fin_test = df_test['date'].max()
print(f"Test DataFrame goes from {init_test} to {fin_test}")

Test DataFrame goes from 2017-08-16 00:00:00 to 2017-08-31 00:00:00


We only have 15 days of testing

## ADDITIONAL DATASETS

### OIL

In [49]:
df_oil = pd.read_csv("../../data/oil.csv")
df_oil.head(5) # Price of the oil during training and testing

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [50]:
fig = px.line(df_oil, x = 'date', y = 'dcoilwtico')
fig.show()

We can observate that some data is missing

In [51]:
print("Percentage of NaN: ", 100*df_oil['dcoilwtico'].isna().sum()/len(df_oil))

Percentage of NaN:  3.5303776683087027


### STORES

In [52]:
df_stores = pd.read_csv('../../data/stores.csv')
df_stores.head(5)

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


Cluster is a grouping of similar stores

In [53]:
df_stores['city'].value_counts()

city
Quito            18
Guayaquil         8
Cuenca            3
Santo Domingo     3
Latacunga         2
Machala           2
Manta             2
Ambato            2
Cayambe           1
Riobamba          1
Ibarra            1
Salinas           1
Puyo              1
Guaranda          1
Quevedo           1
Babahoyo          1
Daule             1
Playas            1
Loja              1
Libertad          1
Esmeraldas        1
El Carmen         1
Name: count, dtype: int64

In [54]:
df_stores['state'].value_counts()

state
Pichincha                         19
Guayas                            11
Azuay                              3
Santo Domingo de los Tsachilas     3
Manabi                             3
Los Rios                           2
Cotopaxi                           2
Tungurahua                         2
El Oro                             2
Bolivar                            1
Imbabura                           1
Chimborazo                         1
Pastaza                            1
Santa Elena                        1
Loja                               1
Esmeraldas                         1
Name: count, dtype: int64

In [55]:
pd.crosstab(df_stores['type'], df_stores['cluster'])

cluster,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
type,,,,,,,,,,,,,,,,,
A,0,0,0,0,1,0,0,0,0,0,3,0,0,4,0,0,1
B,0,0,0,0,0,6,0,0,0,1,0,0,0,0,0,1,0
C,0,0,7,0,0,0,2,0,0,0,0,1,0,0,5,0,0
D,3,2,0,3,0,0,0,3,2,1,0,0,4,0,0,0,0
E,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0


We can see that each cluster only have one type of stores except the cluster 10

### HOLIDAYS

In [56]:
df_holidays = pd.read_csv('../../data/holidays_events.csv')
df_holidays.head(5)

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


A holiday that is transferred officially falls on that calendar day, but was moved to another date by the government. A transferred day is more like a normal day than a holiday. To find the day that it was actually celebrated, look for the corresponding row where type is Transfer.

In [57]:
df_holidays.shape

(350, 6)

In [58]:
df_holidays['locale'].value_counts()

locale
National    174
Local       152
Regional     24
Name: count, dtype: int64

In [59]:
df_holidays.loc[df_holidays['locale'] == 'Regional']['locale_name'].unique()

array(['Cotopaxi', 'Imbabura', 'Santo Domingo de los Tsachilas',
       'Santa Elena'], dtype=object)

We can see that the regions are equivalent to the state level of the stores

In [60]:
df_holidays['type'].value_counts()

type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

### TRANSACTIONS

In [61]:
df_transactions = pd.read_csv('../../data/transactions.csv')
df_transactions.head(5)

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [62]:
# Check for missing values in train data
print("Missing values in train data:")
print(df_train.isnull().sum())
print(f"\nTotal missing values: {df_train.isnull().sum().sum()}")

# Check for missing values in test data
print("\nMissing values in test data:")
print(df_test.isnull().sum())
print(f"\nTotal missing values: {df_test.isnull().sum().sum()}")


Missing values in train data:
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

Total missing values: 0

Missing values in test data:
id             0
date           0
store_nbr      0
family         0
onpromotion    0
dtype: int64

Total missing values: 0


### Date Range Analysis


In [63]:
# Date range analysis
print("Training data date range:")
print(f"Start: {df_train['date'].min()}")
print(f"End: {df_train['date'].max()}")
print(f"Total days: {(df_train['date'].max() - df_train['date'].min()).days}")
print(f"\nTest data date range:")
print(f"Start: {df_test['date'].min()}")
print(f"End: {df_test['date'].max()}")
print(f"Total days: {(df_test['date'].max() - df_test['date'].min()).days}")
print(f"\nGap between train and test: {(df_test['date'].min() - df_train['date'].max()).days} days")


Training data date range:
Start: 2013-01-01 00:00:00
End: 2017-08-15 00:00:00
Total days: 1687

Test data date range:
Start: 2017-08-16 00:00:00
End: 2017-08-31 00:00:00
Total days: 15

Gap between train and test: 1 days


### Sales Statistics


In [64]:
# Sales statistics
print("Sales Statistics:")
print(df_train['sales'].describe())
print(f"\nZero sales count: {(df_train['sales'] == 0).sum()}")
print(f"Zero sales percentage: {100 * (df_train['sales'] == 0).sum() / len(df_train):.2f}%")
print(f"\nSales > 0 statistics:")
print(df_train[df_train['sales'] > 0]['sales'].describe())


Sales Statistics:
count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64

Zero sales count: 939130
Zero sales percentage: 31.30%

Sales > 0 statistics:


count    2.061758e+06
mean     5.207425e+02
std      1.297187e+03
min      1.220000e-01
25%      9.000000e+00
50%      7.846250e+01
75%      3.880000e+02
max      1.247170e+05
Name: sales, dtype: float64


In [65]:
# Sales distribution by family
df_family_stats = df_train.groupby('family')['sales'].agg(['mean', 'median', 'std', 'min', 'max', 'count']).sort_values('mean', ascending=False)
print("Top 10 families by average sales:")
print(df_family_stats.head(10))
print("\nBottom 10 families by average sales:")
print(df_family_stats.tail(10))


Top 10 families by average sales:
                      mean     median          std  min         max  count
family                                                                    
GROCERY I      3776.972100  3185.0000  2874.208845  0.0  124717.000  90936
BEVERAGES      2385.793151  1784.0000  2307.882305  0.0   25413.000  90936
PRODUCE        1349.352123   398.2905  2186.481332  0.0   17850.615  90936
CLEANING       1072.416744   938.0000   734.681493  0.0   11377.000  90936
DAIRY           709.154889   520.0000   671.949638  0.0    5636.000  90936
BREAD/BAKERY    463.336254   401.0000   368.246367  0.0    4551.298  90936
POULTRY         350.532292   205.7430   400.511631  0.0   12143.201  90936
MEATS           341.849965   224.9365   455.908498  0.0   89576.360  90936
PERSONAL CARE   270.432513   222.0000   226.512007  0.0    7504.000  90936
DELI            265.135067   218.9715   210.417073  0.0    2118.325  90936

Bottom 10 families by average sales:
                            

In [66]:
df_train.head(5)

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


### Promotion Analysis


In [67]:

# Promotion statistics
print("Promotion Statistics:")
print(f"Total records with promotion: {(df_train['onpromotion'] == 1).sum()}")
print(f"Promotion percentage: {100 * (df_train['onpromotion'] == 1).sum() / len(df_train):.2f}%")

# Sales comparison: with vs without promotion
sales_with_promo = df_train[df_train['onpromotion'] == 1]['sales']
sales_without_promo = df_train[df_train['onpromotion'] == 0]['sales']

print(f"\nSales with promotion - Mean: {sales_with_promo.mean():.2f}, Median: {sales_with_promo.median():.2f}")
print(f"Sales without promotion - Mean: {sales_without_promo.mean():.2f}, Median: {sales_without_promo.median():.2f}")
print(f"\nPromotion impact: {((sales_with_promo.mean() / sales_without_promo.mean()) - 1) * 100:.2f}% increase")

Promotion Statistics:
Total records with promotion: 174551
Promotion percentage: 5.82%

Sales with promotion - Mean: 467.56, Median: 145.35
Sales without promotion - Mean: 158.25, Median: 3.00

Promotion impact: 195.46% increase


In [68]:
# Promotion impact by family
promo_impact = df_train.groupby(['family', 'onpromotion'])['sales'].mean().unstack()
promo_impact['impact'] = ((promo_impact[1] / promo_impact[0]) - 1) * 100
promo_impact = promo_impact.sort_values('impact', ascending=False)
print("Promotion impact by family (% increase):")
print(promo_impact[['impact']].head(15))

Promotion impact by family (% increase):
onpromotion                      impact
family                                 
SCHOOL AND OFFICE SUPPLIES  1621.868874
BABY CARE                   1414.604792
PET SUPPLIES                 351.982496
HOME AND KITCHEN II          251.818721
HOME APPLIANCES              205.673959
HOME CARE                    203.083516
PRODUCE                      165.499512
PLAYERS AND ELECTRONICS      164.112686
BEAUTY                       162.437739
CELEBRATION                  156.093786
LADIESWEAR                   151.862414
AUTOMOTIVE                   109.666222
HARDWARE                      94.796174
LAWN AND GARDEN               86.500259
HOME AND KITCHEN I            81.542084


In [69]:
# Visualize promotion impact
fig = px.bar(promo_impact.reset_index(), x='family', y='impact', 
             title='Promotion Impact by Family (% Increase)')
fig.update_xaxes(tickangle=45)
fig.update_layout(height=600)
fig.show()


### Time Series Patterns


In [70]:
# Extract time features
df_train['year'] = df_train['date'].dt.year
df_train['month'] = df_train['date'].dt.month
df_train['day'] = df_train['date'].dt.day
df_train['dayofweek'] = df_train['date'].dt.dayofweek
df_train['dayofyear'] = df_train['date'].dt.dayofyear
df_train['week'] = df_train['date'].dt.isocalendar().week
df_train['is_weekend'] = (df_train['dayofweek'] >= 5).astype(int)


In [71]:
# Monthly sales trend
monthly_sales = df_train.groupby('month')['sales'].mean()
fig = px.bar(x=monthly_sales.index, y=monthly_sales.values, 
             title='Average Sales by Month', labels={'x': 'Month', 'y': 'Average Sales'})
fig.show()


In [72]:
# Day of week pattern
dow_sales = df_train.groupby('dayofweek')['sales'].mean()
dow_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
fig = px.bar(x=dow_names, y=dow_sales.values, 
             title='Average Sales by Day of Week', labels={'x': 'Day of Week', 'y': 'Average Sales'})
fig.show()


In [73]:
# Yearly trend
yearly_sales = df_train.groupby('year')['sales'].mean()
fig = px.line(x=yearly_sales.index, y=yearly_sales.values, 
              title='Average Sales Trend by Year', labels={'x': 'Year', 'y': 'Average Sales'})
fig.show()


In [74]:
# Overall sales trend over time
daily_sales = df_train.groupby('date')['sales'].sum()
fig = px.line(x=daily_sales.index, y=daily_sales.values, 
              title='Total Daily Sales Over Time', labels={'x': 'Date', 'y': 'Total Sales'})
fig.show()


In [75]:
# Sales by store
store_sales = df_train.groupby('store_nbr')['sales'].agg(['mean', 'sum', 'count']).sort_values('mean', ascending=False)
print("Top 10 stores by average sales:")
print(store_sales.head(10))
print("\nBottom 10 stores by average sales:")
print(store_sales.tail(10))


Top 10 stores by average sales:
                  mean           sum  count
store_nbr                                  
44         1117.245254  6.208755e+07  55572
45          980.673908  5.449801e+07  55572
47          916.798209  5.094831e+07  55572
3           908.405495  5.048191e+07  55572
49          781.330450  4.342010e+07  55572
46          753.905962  4.189606e+07  55572
48          646.604950  3.593313e+07  55572
51          592.231511  3.291149e+07  55572
8           548.734739  3.049429e+07  55572
50          515.601753  2.865302e+07  55572

Bottom 10 stores by average sales:
                 mean           sum  count
store_nbr                                 
29         175.001038  9.725158e+06  55572
10         172.999096  9.613906e+06  55572
21         166.549808  9.255506e+06  55572
42         160.976173  8.945768e+06  55572
26         139.550887  7.755122e+06  55572
35         138.139340  7.676679e+06  55572
30         132.838006  7.382074e+06  55572
32         107.10

In [76]:
# Merge with store information
df_train_store = df_train.merge(df_stores, on='store_nbr', how='left')

# Sales by store type
sales_by_type = df_train_store.groupby('type')['sales'].mean()
fig = px.bar(x=sales_by_type.index, y=sales_by_type.values, 
             title='Average Sales by Store Type', labels={'x': 'Store Type', 'y': 'Average Sales'})
fig.show()
